# THESIS-001 — VM Setup (UTM)
**Story Points:** 3 | **Status:** TODO

Set up a UTM VM: Ubuntu 22.04, 4 vCPU, 8 GB RAM.
Provision via Ansible: Python 3.12, Docker CE, PostgreSQL 16, Dagster 1.12.7.


## Acceptance Criteria
- [ ] UTM VM Ubuntu 22.04 running, SSH reachable
- [ ] 4 vCPU / 8 GB RAM confirmed
- [ ] Python 3.12 installed
- [ ] Docker CE running
- [ ] PostgreSQL 16 running
- [ ] Dagster 1.12.7 installed, gRPC on port 4000

## Step 1 — Write VM IP to vm-ip.txt

In [ ]:
import os
vm_ip_file = '../vm-ip.txt'
if os.path.exists(vm_ip_file):
    print('VM IP:', open(vm_ip_file).read().strip())
else:
    print('vm-ip.txt not found — create the VM in UTM first, then write its IP here.')

## Step 2 — Provision via Ansible

In [ ]:
import subprocess
r = subprocess.run(['make', '-C', '..', 'vm-provision'], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])

## Step 3 — Validate VM

In [ ]:
import subprocess, os
vm_ip_file = '../vm-ip.txt'
if not os.path.exists(vm_ip_file):
    print('vm-ip.txt not found')
else:
    ip = open(vm_ip_file).read().strip()
    for cmd, label in [
        ('nproc', 'CPU count'),
        ('free -h | head -2', 'Memory'),
        ('python3 --version', 'Python'),
        ('docker --version', 'Docker CE'),
        ('pg_isready', 'PostgreSQL'),
        ('dagster --version', 'Dagster'),
    ]:
        r = subprocess.run(
            ['ssh', '-i', os.path.expanduser('~/.ssh/thesis_vm'),
             '-o', 'StrictHostKeyChecking=no', f'ubuntu@{ip}', cmd],
            capture_output=True, text=True, timeout=10
        )
        icon = 'OK' if r.returncode == 0 else '!!'
        print(f'  {icon}  {label:<15} {r.stdout.strip() or r.stderr.strip()}')